# Explore the event stream

The raw event stream is the input to every stage of the pipeline.
Before writing a single line of transformation code, look at what
actually arrives.

This notebook is the equivalent of opening the Pub/Sub subscription in
the console and reading the first hundred messages. In a real project
you would do this against the live topic; here it runs against the
deterministic sample dataset, so every number in the output is one the
course asserts against.

Run it with `jupyter lab notebooks/` from the course root.

## 0. Setup

Put the course's pure core modules on the path. No installation
and no cloud SDK — that is the whole point of the split.

In [ ]:
import sys
from pathlib import Path

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project" / "core").exists():
    COURSE_ROOT = Path.cwd().parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))

SAMPLE_DIR = COURSE_ROOT / "sample_data"


## 1. Load the raw rows

The file is CSV, but the pipeline treats every row as a
*message* — the same shape Pub/Sub hands you.

In [ ]:
from pipeline_logic import ingest_events

lines = (SAMPLE_DIR / 'events.csv').read_text(encoding='utf-8').splitlines()
print(f'raw lines: {len(lines)}')
print(lines[0])
print(lines[1])

## 2. Ingest and split

`ingest_events` returns two lists: the rows that passed
validation and the rows that were rejected with a reason code.
The split is the whole point of a landing zone — bad data goes
somewhere specific.

In [ ]:
accepted, rejected = ingest_events(lines)
print(f'accepted: {len(accepted)}')
print(f'rejected: {len(rejected)}')
accepted[0]

## 3. What is in a row?

One normalised row. Every field is flat and primitive, because a
BigQuery table with nested fields costs more to scan and more to
reason about.

In [ ]:
row = accepted[0]
for key, value in row.items():
    print(f'{key:<16} {value!r}')

## 4. The cardinality of each field

Low-cardinality fields are candidates for clustering.
High-cardinality fields are candidates for a bug. Run this before
you design the table, not after.

In [ ]:
for field in ('region', 'channel', 'currency', 'event_type'):
    values = {r[field] for r in accepted}
    print(f'{field:<12} {len(values):>3} distinct  {sorted(values)}')

## 5. The event-time span

The windowing path derives the bucket from the **event timestamp**,
never from arrival time. So the event-time span is what determines
how many windows exist.

In [ ]:
from pipeline_logic import event_time_bounds, hours_spanned

earliest, latest = event_time_bounds(accepted)
print('earliest:', earliest)
print('latest:  ', latest)
print('hours spanned:', hours_spanned(accepted))